# Clustering Analysis — Whisper Encoder Representations
Visualisation and analysis of phoneme / L1 / speaker clustering metrics across encoder layers.
Loads per-model JSON files (`clustering_{model_key}_{split}.json`).

In [1]:
# ── Config ────────────────────────────────────────────────────────────────────
import json, re
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, HTML
import ipywidgets as widgets
from pathlib import Path

SPLIT      = "scripted"
PLOTS_DIR  = Path("results/clustering")
LABEL_TYPES = ["phoneme", "l1", "speaker"]
METRICS = {
    "silhouette":        ("Silhouette ↑",         "higher is better"),
    "davies_bouldin":    ("Davies-Bouldin ↓",      "lower is better"),
    "calinski_harabasz": ("Calinski-Harabasz ↑",   "higher is better"),
    "wb_ratio":          ("Within/Between Ratio ↓", "lower is better"),
}

MODEL_STYLES = {
    "baseline":      {"label": "Baseline (frozen)", "color": "#636EFA", "dash": "solid"},
    "baseline_lora": {"label": "Naive LoRA FT",     "color": "#EF553B", "dash": "dash"},
    "ctc_aux":       {"label": "CTC Aux",            "color": "#00CC96", "dash": "dot"},
}

In [2]:
# ── Load all available per-model JSONs ────────────────────────────────────────
raw = {}
for key in MODEL_STYLES:
    p = PLOTS_DIR / f"clustering_{key}_{SPLIT}.json"
    if not p.exists():
        print(f"  [missing] {p}")
        continue
    with open(p) as f:
        raw[key] = json.load(f)
    print(f"  Loaded {key}: layers {sorted(int(k) for k in raw[key].keys())}")

MODELS = list(raw.keys())
LAYERS = sorted({int(k) for d in raw.values() for k in d.keys()})
print(f"\nActive models : {MODELS}")
print(f"Layers        : {LAYERS}")

  Loaded baseline: layers [0, 1, 2, 3, 4, 5, 6]
  Loaded baseline_lora: layers [0, 1, 2, 3, 4, 5, 6]
  Loaded ctc_aux: layers [0, 1, 2, 3, 4, 5, 6]

Active models : ['baseline', 'baseline_lora', 'ctc_aux']
Layers        : [0, 1, 2, 3, 4, 5, 6]


In [3]:
# ── Flatten to DataFrame ──────────────────────────────────────────────────────
rows = []
for key, model_data in raw.items():
    for layer in LAYERS:
        for label_type in LABEL_TYPES:
            d = model_data.get(str(layer), {}).get(label_type, {})
            if "error" in d:
                continue
            rows.append({
                "model": key, "layer": layer, "label_type": label_type,
                **{m: d.get(m) for m in METRICS},
                "within_mean": d.get("within_mean"),
                "between_mean": d.get("between_mean"),
                "n_samples": d.get("n_samples"),
                "n_classes": d.get("n_classes"),
            })

df = pd.DataFrame(rows)
print(f"Loaded {len(df)} metric rows")
display(df.head())

Loaded 63 metric rows


,model,layer,label_type,silhouette,davies_bouldin,calinski_harabasz,wb_ratio,within_mean,between_mean,n_samples,n_classes
0,baseline,0,phoneme,-0.087889,8.496769,11.631238,2.150298,25.632765,11.920566,40251,37
1,baseline,0,l1,-0.010229,17.071792,8.702700,6.917310,27.242943,3.938373,40251,6
2,baseline,0,speaker,-0.036198,9.721840,9.405299,3.325116,26.805639,8.061566,40251,24
3,baseline,1,phoneme,-0.032235,7.267144,16.963499,1.828022,25.224689,13.798896,40251,37
4,baseline,1,l1,-0.003119,14.850149,11.865104,5.992068,27.249083,4.547526,40251,6


---
# Section 1 — UMAP Plots

In [4]:
def _img_box(path, label):
    if Path(path).exists():
        img = widgets.Image(value=open(path, "rb").read(), format="png",
                            layout=widgets.Layout(width="100%"))
        return widgets.VBox(
            [widgets.HTML(f'<b style="font-size:11px;text-align:center">{label}</b>'), img],
            layout=widgets.Layout(align_items="center", width="100%"),
        )
    return widgets.Label(f"Missing: {Path(path).name}")

def show_umap_row(model_keys, layer, plot_type):
    """One row: all models side-by-side for a given layer + plot_type."""
    boxes = []
    for key in model_keys:
        path  = PLOTS_DIR / key / f"umap_{key}_layer{layer}_{plot_type}.png"
        label = f"{MODEL_STYLES.get(key, {}).get('label', key)} | L{layer}"
        boxes.append(_img_box(str(path), label))
    if any(isinstance(b, widgets.VBox) for b in boxes):
        display(widgets.HBox(boxes, layout=widgets.Layout(
            justify_content="space-around", margin="4px 0")))

umap_layers = [0, 3, 6]

for plot_type, heading in [
    ("l1",                "L1 / Accent Clusters"),
    ("phoneme",           "Phoneme Clusters"),
    ("focus_phones",      "Focus Phones (θ ð v f s z)"),
    ("focus_phones_l1",   "Focus Phones — coloured by L1"),
]:
    any_exist = any(
        (PLOTS_DIR / k / f"umap_{k}_layer{li}_{plot_type}.png").exists()
        for k in MODELS for li in umap_layers
    )
    if not any_exist:
        continue
    display(HTML(f"<h3>{heading}</h3>"))
    for layer in umap_layers:
        display(HTML(f"<b>Layer {layer}</b>"))
        show_umap_row(MODELS, layer, plot_type)
    display(HTML('<hr style="border:0.5px solid #ddd"/>'))

---
# Section 2 — Metric Tables

In [5]:
for label_type in LABEL_TYPES:
    display(HTML(f"<h3>Label: <code>{label_type}</code></h3>"))
    sub = df[df["label_type"] == label_type].copy()
    for metric, (title, note) in METRICS.items():
        pivot = sub.pivot(index="layer", columns="model", values=metric)
        pivot.columns = [MODEL_STYLES.get(c, {}).get("label", c) for c in pivot.columns]
        pivot.index.name = "Layer"
        cmap = "RdYlGn" if "↑" in title else "RdYlGn_r"
        display(HTML(f'<b>{title}</b> <span style="color:#888;font-size:12px">({note})</span>'))
        display(pivot.style.format("{:.4f}").background_gradient(cmap=cmap, axis=None))
    display(HTML("<hr/>"))

,Baseline (frozen),Naive LoRA FT,CTC Aux
Layer,,,
0,-0.0879,-0.0806,-0.1094
1,-0.0322,-0.0381,-0.0487
2,-0.0283,-0.0300,-0.0453
3,-0.0152,-0.0241,-0.0227
4,-0.0011,-0.0001,-0.0089
5,0.0162,0.0079,0.0070
6,0.0265,0.0195,0.0171


,Baseline (frozen),Naive LoRA FT,CTC Aux
Layer,,,
0,8.4968,8.9321,8.6195
1,7.2671,7.3122,7.2215
2,6.6951,6.4955,7.0206
3,6.2733,6.2103,6.3916
4,6.0060,6.0160,6.1641
5,5.4502,5.4696,5.5260
6,5.2919,5.2517,5.5249


,Baseline (frozen),Naive LoRA FT,CTC Aux
Layer,,,
0,11.6312,11.4614,11.6054
1,16.9635,16.2545,16.5510
2,19.6091,18.8786,18.4571
3,19.6831,19.3698,19.4510
4,20.1044,19.7132,19.1962
5,22.6570,22.7321,21.2775
6,21.0601,21.3585,20.1549


,Baseline (frozen),Naive LoRA FT,CTC Aux
Layer,,,
0,2.1503,2.1777,2.1527
1,1.8280,1.8644,1.8426
2,1.7118,1.7368,1.7519
3,1.6874,1.6906,1.6874
4,1.6520,1.6814,1.6935
5,1.5735,1.5524,1.6054
6,1.5987,1.5789,1.6501


,Baseline (frozen),Naive LoRA FT,CTC Aux
Layer,,,
0,-0.0102,-0.0185,-0.0174
1,-0.0031,-0.0037,-0.0060
2,-0.0006,0.0006,-0.0036
3,0.0014,-0.0001,-0.0016
4,0.0004,0.0015,-0.0012
5,0.0025,0.0008,0.0007
6,0.0073,0.0075,0.0072


...

,Baseline (frozen),Naive LoRA FT,CTC Aux
Layer,,,
0,6.9173,6.5900,6.5960
1,5.9921,6.0651,5.8510
2,5.7862,5.7194,5.9298
3,5.9902,5.8479,6.1444
4,6.2552,6.2344,6.2196
5,6.0965,5.8483,5.9031
6,5.2970,5.1257,5.3079


,Baseline (frozen),Naive LoRA FT,CTC Aux
Layer,,,
0,-0.0362,-0.0278,-0.0311
1,-0.0040,-0.0038,-0.0089
2,0.0023,0.0024,-0.0044
3,-0.0013,0.0021,0.0009
4,-0.0013,-0.0044,-0.0046
5,-0.0022,-0.0012,-0.0011
6,-0.0010,-0.0002,-0.0026


,Baseline (frozen),Naive LoRA FT,CTC Aux
Layer,,,
0,9.7218,9.6791,9.6839
1,8.6437,8.5561,8.5922
2,8.3130,8.2350,8.5737
3,8.7246,8.7945,8.8592
4,9.3423,9.4489,9.1518
5,9.5986,9.6233,9.3647
6,10.4627,10.4099,10.3415


,Baseline (frozen),Naive LoRA FT,CTC Aux
Layer,,,
0,9.4053,9.3743,9.2381
1,13.5105,13.9266,15.7241
2,12.3210,13.0697,13.3574
3,10.0145,10.0334,10.9738
4,8.2128,8.4947,9.7986
5,7.8360,7.7469,8.7019
6,7.5294,7.7472,8.0204


,Baseline (frozen),Naive LoRA FT,CTC Aux
Layer,,,
0,3.3251,3.3311,3.3556
1,2.8097,2.7622,2.6207
2,2.9154,2.8347,2.8064
3,3.2269,3.2139,3.0885
4,3.5479,3.4909,3.2707
5,3.6356,3.6443,3.4595
6,3.7355,3.6889,3.6208


---
# Section 3 — Layer Progression Charts

In [6]:
# ── 3a. Per-label metric progression ─────────────────────────────────────────
for label_type in LABEL_TYPES:
    sub = df[df["label_type"] == label_type]
    fig = make_subplots(rows=2, cols=2,
                        subplot_titles=[t for t, _ in METRICS.values()])
    positions = [(1,1),(1,2),(2,1),(2,2)]
    for (metric, (title, _)), (r, c) in zip(METRICS.items(), positions):
        for key in MODELS:
            msub = sub[sub["model"] == key].sort_values("layer")
            s    = MODEL_STYLES.get(key, {})
            fig.add_trace(go.Scatter(
                x=msub["layer"], y=msub[metric],
                name=s.get("label", key), legendgroup=key,
                showlegend=(r == 1 and c == 1),
                line=dict(color=s.get("color"), dash=s.get("dash"), width=2),
                mode="lines+markers",
            ), row=r, col=c)
        fig.update_xaxes(title_text="Layer", row=r, col=c)
        fig.update_yaxes(title_text=metric[:14], row=r, col=c)
    fig.update_layout(
        title=f"Clustering Metrics by Layer — {label_type}",
        height=600,
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=0.5),
    )
    fig.show()

In [7]:
# ── 3b. Silhouette cross-label comparison (all three on same axes per model) ──
LABEL_COLORS = {"phoneme": "#00CC96", "l1": "#AB63FA", "speaker": "#FFA15A"}
LABEL_DASH   = {"phoneme": "solid",   "l1": "dot",     "speaker": "dash"}

fig = make_subplots(rows=1, cols=len(MODELS),
                    subplot_titles=[MODEL_STYLES.get(k, {}).get("label", k) for k in MODELS],
                    shared_yaxes=True)
for ci, key in enumerate(MODELS):
    for lt in LABEL_TYPES:
        sub = df[(df["model"] == key) & (df["label_type"] == lt)].sort_values("layer")
        fig.add_trace(go.Scatter(
            x=sub["layer"], y=sub["silhouette"],
            name=lt, legendgroup=lt, showlegend=(ci == 0),
            mode="lines+markers",
            line=dict(color=LABEL_COLORS[lt], dash=LABEL_DASH[lt], width=2),
        ), row=1, col=ci + 1)
fig.add_hline(y=0, line_dash="dot", line_color="gray", line_width=1)
fig.update_xaxes(title_text="Layer")
fig.update_yaxes(title_text="Silhouette Score", col=1)
fig.update_layout(
    title="Silhouette Score by Layer: Phoneme vs L1 vs Speaker",
    height=420,
    legend=dict(orientation="h", yanchor="bottom", y=1.05, xanchor="center", x=0.5),
)
fig.show()

---
# Section 4 — Delta: Fine-tuned vs Baseline

In [8]:
# For each non-baseline model, show LoRA-improvement delta tables + bar charts
for ft_key in [k for k in MODELS if k != "baseline"]:
    if "baseline" not in raw:
        continue
    ft_label = MODEL_STYLES.get(ft_key, {}).get("label", ft_key)
    display(HTML(f"<h3>{ft_label} − Baseline Delta (positive = improvement)</h3>"))

    for label_type in LABEL_TYPES:
        base = df[(df["model"] == "baseline") & (df["label_type"] == label_type)].set_index("layer")
        ft   = df[(df["model"] == ft_key)     & (df["label_type"] == label_type)].set_index("layer")
        if ft.empty:
            continue
        delta = pd.DataFrame(index=LAYERS)
        for metric, (title, note) in METRICS.items():
            sign = -1 if "↓" in note else 1
            delta[title] = sign * (ft[metric] - base[metric])
        delta.index.name = "Layer"
        display(HTML(f"<b>{label_type}</b>"))
        display(delta.style.format("{:+.4f}")
                .background_gradient(cmap="RdYlGn", axis=None))

    # Bar chart: silhouette delta by label_type
    fig = go.Figure()
    for lt in LABEL_TYPES:
        base = df[(df["model"] == "baseline") & (df["label_type"] == lt)].sort_values("layer")
        ft   = df[(df["model"] == ft_key)     & (df["label_type"] == lt)].sort_values("layer")
        if ft.empty:
            continue
        delta = ft["silhouette"].values - base["silhouette"].values
        fig.add_trace(go.Bar(
            x=LAYERS, y=delta.tolist(), name=lt,
            marker_color=LABEL_COLORS[lt],
        ))
    fig.add_hline(y=0, line_color="black", line_width=0.8)
    fig.update_layout(
        title=f"Silhouette Δ by Layer — {ft_label} vs Baseline",
        barmode="group", height=380,
        legend=dict(orientation="h", y=1.05, xanchor="center", x=0.5),
    )
    fig.update_xaxes(title_text="Layer")
    fig.update_yaxes(title_text="Δ Silhouette")
    fig.show()

,Silhouette ↑,Davies-Bouldin ↓,Calinski-Harabasz ↑,Within/Between Ratio ↓
Layer,,,,
0,+0.0073,+0.4354,-0.1699,+0.0274
1,-0.0058,+0.0451,-0.7090,+0.0364
2,-0.0017,-0.1996,-0.7305,+0.0250
3,-0.0089,-0.0630,-0.3133,+0.0032
4,+0.0010,+0.0101,-0.3913,+0.0295
5,-0.0083,+0.0194,+0.0751,-0.0212
6,-0.0071,-0.0401,+0.2984,-0.0198


,Silhouette ↑,Davies-Bouldin ↓,Calinski-Harabasz ↑,Within/Between Ratio ↓
Layer,,,,
0,-0.0083,-0.2761,+1.1017,-0.3273
1,-0.0006,+0.0637,-0.2643,+0.0730
2,+0.0012,-0.0048,+0.1725,-0.0668
3,-0.0015,-0.3976,+0.6440,-0.1423
4,+0.0011,+0.2370,+0.1674,-0.0207
5,-0.0016,-0.4620,+1.1730,-0.2482
6,+0.0002,+0.0910,+1.0902,-0.1713


,Silhouette ↑,Davies-Bouldin ↓,Calinski-Harabasz ↑,Within/Between Ratio ↓
Layer,,,,
0,+0.0084,-0.0428,-0.0310,+0.0059
1,+0.0003,-0.0877,+0.4160,-0.0475
2,+0.0001,-0.0780,+0.7486,-0.0808
3,+0.0035,+0.0699,+0.0189,-0.0130
4,-0.0032,+0.1067,+0.2819,-0.0571
5,+0.0010,+0.0247,-0.0891,+0.0087
6,+0.0008,-0.0528,+0.2177,-0.0465


,Silhouette ↑,Davies-Bouldin ↓,Calinski-Harabasz ↑,Within/Between Ratio ↓
Layer,,,,
0,-0.0215,+0.1227,-0.0259,+0.0024
1,-0.0165,-0.0457,-0.4125,+0.0146
2,-0.0170,+0.3255,-1.1520,+0.0401
3,-0.0075,+0.1183,-0.2321,-0.0000
4,-0.0078,+0.1582,-0.9083,+0.0415
5,-0.0092,+0.0759,-1.3795,+0.0319
6,-0.0094,+0.2330,-0.9051,+0.0514


,Silhouette ↑,Davies-Bouldin ↓,Calinski-Harabasz ↑,Within/Between Ratio ↓
Layer,,,,
0,-0.0072,+0.0331,+1.1341,-0.3213
1,-0.0029,-0.0951,+0.4130,-0.1411
2,-0.0030,+0.9228,-0.5187,+0.1436
3,-0.0029,+0.2943,-0.2916,+0.1542
4,-0.0017,+0.5809,+0.1864,-0.0356
5,-0.0017,-0.6605,+0.8263,-0.1934
6,-0.0000,+0.0548,-0.0892,+0.0109


,Silhouette ↑,Davies-Bouldin ↓,Calinski-Harabasz ↑,Within/Between Ratio ↓
Layer,,,,
0,+0.0051,-0.0379,-0.1672,+0.0305
1,-0.0049,-0.0515,+2.2136,-0.1891
2,-0.0067,+0.2607,+1.0364,-0.1091
3,+0.0023,+0.1346,+0.9593,-0.1384
4,-0.0034,-0.1904,+1.5858,-0.2772
5,+0.0011,-0.2339,+0.8659,-0.1760
6,-0.0016,-0.1212,+0.4910,-0.1146


---
# Section 5 — Deeper Diagnostics

In [9]:
# ── 5a. Accent-Phoneme Entanglement Index ─────────────────────────────────────
display(HTML("<h4>Accent-Phoneme Entanglement Index (L1 sil / |phoneme sil|)</h4>"))
display(HTML('<p style="color:#888">Higher = accent more structured than phoneme = more entangled</p>'))

fig = go.Figure()
for key in MODELS:
    ph = df[(df["model"] == key) & (df["label_type"] == "phoneme")].sort_values("layer")
    l1 = df[(df["model"] == key) & (df["label_type"] == "l1")].sort_values("layer")
    ratio = l1["silhouette"].values / (np.abs(ph["silhouette"].values) + 1e-8)
    s = MODEL_STYLES.get(key, {})
    fig.add_trace(go.Scatter(
        x=LAYERS, y=ratio.tolist(), name=s.get("label", key),
        mode="lines+markers",
        line=dict(color=s.get("color"), dash=s.get("dash"), width=2),
    ))
fig.update_layout(title="Accent-Phoneme Entanglement Index", height=380,
                  legend=dict(orientation="h", y=1.05, xanchor="center", x=0.5))
fig.update_xaxes(title_text="Layer")
fig.update_yaxes(title_text="L1 sil / |phoneme sil|")
fig.show()

In [10]:
# ── 5b. L1 − Speaker silhouette: group-level vs speaker-level accent ──────────
display(HTML("<h4>L1 − Speaker Silhouette Delta</h4>"))
display(HTML('<p style="color:#888">'
             "Positive = accent encoded as group-level structure beyond speaker identity. "
             "Near zero = accent is a proxy for speaker ID.</p>"))

fig = go.Figure()
for key in MODELS:
    l1  = df[(df["model"] == key) & (df["label_type"] == "l1")].sort_values("layer")
    spk = df[(df["model"] == key) & (df["label_type"] == "speaker")].sort_values("layer")
    delta = l1["silhouette"].values - spk["silhouette"].values
    s = MODEL_STYLES.get(key, {})
    fig.add_trace(go.Scatter(
        x=LAYERS, y=delta.tolist(), name=s.get("label", key),
        mode="lines+markers",
        line=dict(color=s.get("color"), dash=s.get("dash"), width=2),
    ))
fig.add_hline(y=0, line_dash="dot", line_color="gray")
fig.update_layout(title="L1 − Speaker Silhouette Delta by Layer", height=380,
                  legend=dict(orientation="h", y=1.05, xanchor="center", x=0.5))
fig.update_xaxes(title_text="Layer")
fig.update_yaxes(title_text="Δ Silhouette (L1 − Speaker)")
fig.show()

In [11]:
# ── 5c. Best-layer radar chart ────────────────────────────────────────────────
display(HTML("<h4>Best-Layer Metric Summary — Radar</h4>"))
radar_metrics = list(METRICS.keys())
radar_labels  = [t for t, _ in METRICS.values()]

fig = go.Figure()
for key in MODELS:
    for lt in ["phoneme", "l1"]:
        sub  = df[(df["model"] == key) & (df["label_type"] == lt)]
        vals = []
        for metric, (_, note) in METRICS.items():
            vals.append(float(sub[metric].min() if "↓" in note else sub[metric].max()))
        fig.add_trace(go.Scatterpolar(
            r=vals + [vals[0]], theta=radar_labels + [radar_labels[0]],
            name=f"{MODEL_STYLES.get(key,{}).get('label',key)}/{lt}",
            fill="toself", opacity=0.25,
        ))
fig.update_layout(
    polar=dict(radialaxis=dict(visible=True)),
    title="Best-Layer Metric Summary (raw values — axes not comparable across metrics)",
    height=500,
    legend=dict(orientation="h", y=-0.2, xanchor="center", x=0.5),
)
fig.show()

---
# Section 6 — Numerical Summary

In [12]:
pd.set_option("display.float_format", "{:.4f}".format)
out_path = Path("results/clustering_summary.csv")
df.to_csv(out_path, index=False)
print(f"Saved: {out_path}")
display(df.sort_values(["model", "label_type", "layer"]))

Saved: results/clustering_summary.csv


,model,layer,label_type,silhouette,davies_bouldin,calinski_harabasz,wb_ratio,within_mean,between_mean,n_samples,n_classes
1,baseline,0,l1,-0.0102,17.0718,8.7027,6.9173,27.2429,3.9384,40251,6
4,baseline,1,l1,-0.0031,14.8501,11.8651,5.9921,27.2491,4.5475,40251,6
7,baseline,2,l1,-0.0006,13.9642,12.5712,5.7862,27.2436,4.7084,40251,6
10,baseline,3,l1,0.0014,14.6035,11.5673,5.9902,27.2702,4.5525,40251,6
13,baseline,4,l1,0.0004,14.8597,10.7481,6.2552,27.2733,4.3601,40251,6
...,...,...,...,...,...,...,...,...,...,...,...
50,ctc_aux,2,speaker,-0.0044,8.5737,13.3574,2.8064,26.5925,9.4757,40251,24
53,ctc_aux,3,speaker,0.0009,8.8592,10.9738,3.0885,26.7342,8.6560,40251,24
56,ctc_aux,4,speaker,-0.0046,9.1518,9.7986,3.2707,26.8184,8.1996,40251,24
59,ctc_aux,5,speaker,-0.0011,9.3647,8.7019,3.4595,26.8360,7.7571,40251,24
